In [ ]:
!pip install -q torch torchvision wandb datasets huggingface_hub scikit-learn matplotlib

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from datasets import load_dataset
from sklearn.metrics import confusion_matrix
from huggingface_hub import HfApi, hf_hub_download
from PIL import Image
import wandb

## Config & Authentication

In [4]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
WANDB_API_KEY = userdata.get("WANDB_API_KEY")
HF_REPO_ID = "Tron2703/stl10-resnet18"

os.environ["WANDB_API_KEY"] = WANDB_API_KEY

EPOCHS = 10
BATCH_SIZE = 64
LR = 1e-3
NUM_WORKERS = 2
PROJECT_NAME = "stl10-resnet18"
CHECKPOINT_PATH = "best_model.pth"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Dataset & DataLoaders

In [5]:
CLASS_NAMES = [
    "airplane", "bird", "car", "cat", "deer",
    "dog", "horse", "monkey", "ship", "truck"
]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [6]:
class STL10Dataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item["image"]
        label = item["label"]

        if not isinstance(image, Image.Image):
            image = Image.fromarray(image)
        if image.mode != "RGB":
            image = image.convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
dataset = load_dataset("Chiranjeev007/STL-10_Subset")

train_dataset = STL10Dataset(dataset["train"], transform=train_transforms)
val_dataset = STL10Dataset(dataset["validation"], transform=eval_transforms)
test_dataset = STL10Dataset(dataset["test"], transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

README.md:   0%|          | 0.00/723 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/88.9M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/8.82M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train: 5000 | Val: 500 | Test: 1000


## Model Setup

In [8]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 126MB/s]


## Training

In [9]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, correct / total

In [10]:
wandb.init(project=PROJECT_NAME, config={
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "model": "resnet18",
    "dataset": "STL-10_Subset",
})

best_val_acc = 0.0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    wandb.log({
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "train/accuracy": train_acc,
        "val/loss": val_loss,
        "val/accuracy": val_acc,
        "lr": optimizer.param_groups[0]["lr"],
    })

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  -> Best model saved (val_acc: {val_acc:.4f})")

wandb.finish()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: b22cs093 (b22cs093-prom-iit-rajasthan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/10 | Train Loss: 1.3282 Acc: 0.5266 | Val Loss: 1.2466 Acc: 0.5820
  -> Best model saved (val_acc: 0.5820)
Epoch 2/10 | Train Loss: 1.0377 Acc: 0.6354 | Val Loss: 1.0382 Acc: 0.6680
  -> Best model saved (val_acc: 0.6680)
Epoch 3/10 | Train Loss: 0.9213 Acc: 0.6780 | Val Loss: 0.9259 Acc: 0.6920
  -> Best model saved (val_acc: 0.6920)
Epoch 4/10 | Train Loss: 0.8484 Acc: 0.6924 | Val Loss: 0.7172 Acc: 0.7460
  -> Best model saved (val_acc: 0.7460)
Epoch 5/10 | Train Loss: 0.7904 Acc: 0.7246 | Val Loss: 0.5494 Acc: 0.8040
  -> Best model saved (val_acc: 0.8040)
Epoch 6/10 | Train Loss: 0.6517 Acc: 0.7728 | Val Loss: 0.5399 Acc: 0.8160
  -> Best model saved (val_acc: 0.8160)
Epoch 7/10 | Train Loss: 0.5729 Acc: 0.8006 | Val Loss: 0.4068 Acc: 0.8600
  -> Best model saved (val_acc: 0.8600)
Epoch 8/10 | Train Loss: 0.4800 Acc: 0.8340 | Val Loss: 0.3444 Acc: 0.8780
  -> Best model saved (val_acc: 0.8780)
Epoch 9/10 | Train Loss: 0.4276 Acc: 0.8482 | Val Loss: 0.3254 Acc: 0.8760
Epoch

epoch,▁▂▃▃▄▅▆▆▇█
lr,█▇▇▆▅▃▂▂▁▁
train/accuracy,▁▃▄▅▅▆▇███
train/loss,█▆▅▄▄▃▂▂▁▁
val/accuracy,▁▃▄▅▆▆▇███
val/loss,█▆▆▄▃▃▂▁▁▁
epoch,10
lr,0
train/accuracy,0.8566
train/loss,0.41239
val/accuracy,0.886


## Push Best Model to HuggingFace Hub

In [11]:
api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, token=HF_TOKEN, exist_ok=True)
api.upload_file(
    path_or_fileobj=CHECKPOINT_PATH,
    path_in_repo="best_model.pth",
    repo_id=HF_REPO_ID,
    token=HF_TOKEN,
)
print(f"Best model pushed to https://huggingface.co/{HF_REPO_ID}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  best_model.pth              :   1%|1         |  567kB / 44.8MB            

Best model pushed to https://huggingface.co/Tron2703/stl10-resnet18


## Load Model from HuggingFace Hub & Evaluate on Test Set

In [12]:
model_path = hf_hub_download(repo_id=HF_REPO_ID, filename="best_model.pth", token=HF_TOKEN)
hf_model = models.resnet18(weights=None)
hf_model.fc = nn.Linear(hf_model.fc.in_features, len(CLASS_NAMES))
hf_model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
hf_model = hf_model.to(device)
hf_model.eval()
print("Model loaded from HuggingFace Hub")

best_model.pth:   0%|          | 0.00/44.8M [00:00<?, ?B/s]

Model loaded from HuggingFace Hub


In [13]:
@torch.no_grad()
def get_predictions(model, loader, device):
    all_preds = []
    all_labels = []
    all_images = []

    for images, labels in loader:
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_images.extend(images.cpu())

    return np.array(all_preds), np.array(all_labels), all_images


def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    tensor = tensor.cpu() * std + mean
    return tensor.clamp(0, 1)


preds, labels, images = get_predictions(hf_model, test_loader, device)
overall_acc = (preds == labels).mean()
print(f"Test Accuracy: {overall_acc:.4f}")

Test Accuracy: 0.9200


## Log Confusion Matrix to W&B

In [14]:
wandb.init(project=PROJECT_NAME, job_type="evaluation")

wandb.log({"test/accuracy": overall_acc})

wandb.log({
    "confusion_matrix": wandb.plot.confusion_matrix(
        probs=None,
        y_true=labels.tolist(),
        preds=preds.tolist(),
        class_names=CLASS_NAMES,
    )
})

## Log Class-wise Accuracy Bar Plot to W&B

In [15]:
cm = confusion_matrix(labels, preds)
class_acc = cm.diagonal() / cm.sum(axis=1)

table = wandb.Table(
    data=[[name, acc] for name, acc in zip(CLASS_NAMES, class_acc)],
    columns=["Class", "Accuracy"]
)
wandb.log({
    "class_wise_accuracy": wandb.plot.bar(
        table, "Class", "Accuracy", title="Class-wise Accuracy"
    )
})

## Log 20 Test Samples (10 Correct + 10 Incorrect) to W&B

In [16]:
correct_idx = np.where(preds == labels)[0]
incorrect_idx = np.where(preds != labels)[0]

np.random.seed(42)
correct_samples = np.random.choice(correct_idx, min(10, len(correct_idx)), replace=False)
incorrect_samples = np.random.choice(incorrect_idx, min(10, len(incorrect_idx)), replace=False)

sample_table = wandb.Table(columns=["Image", "Predicted", "Actual", "Correct"])

for idx in correct_samples:
    img_tensor = denormalize(images[idx])
    img = Image.fromarray((img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8))
    sample_table.add_data(wandb.Image(img), CLASS_NAMES[preds[idx]], CLASS_NAMES[labels[idx]], "Yes")

for idx in incorrect_samples:
    img_tensor = denormalize(images[idx])
    img = Image.fromarray((img_tensor.permute(1, 2, 0).numpy() * 255).astype(np.uint8))
    sample_table.add_data(wandb.Image(img), CLASS_NAMES[preds[idx]], CLASS_NAMES[labels[idx]], "No")

wandb.log({"test_predictions": sample_table})

wandb.finish()
print("Evaluation complete! Check your W&B dashboard.")

test/accuracy,▁
test/accuracy,0.92


Evaluation complete! Check your W&B dashboard.
